In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

cfg = common.load_config("dev")
cat = cfg["catalog"]
m = spark.table(f"{cat}.silver.customer_id_map")
ids, survivors = m.count(), m.filter("is_survivor").count()
display(m.filter("group_size > 1").orderBy("survivor_id", "is_survivor"))
# the same person under two ids, as the source wrote them
display(spark.sql(f"""
    SELECT m.survivor_id, c.customer_id, c.full_name, c.branch_id, c.valid_from
    FROM {cat}.silver.customer_id_map m
    JOIN {cat}.silver.customers c ON c.customer_id = m.customer_id AND c.is_current
    WHERE m.group_size > 1 ORDER BY m.survivor_id, c.valid_from"""))
print("ids", ids, "| real customers", survivors, "| duplicates", ids - survivors)
print("PASS" if (ids, survivors) == (119, 116) else "FAIL")